In [32]:
import os
import pandas as pd
import tensorflow as tf
from pathlib import Path
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

In [3]:
print(Path.cwd().parent)

/Users/finnerty/Documents/Swinburne/FinalSem/MachineSystems/AMLFacialRecognitionProject


In [ ]:
PROJECT_ROOT =  Path.cwd().parent

DATASET = (
    PROJECT_ROOT
    / "data"
    / "lcc-fasd-casia"
    / "LCC_FASD"
)

TRAIN_IMG_DIR = DATASET / "LCC_FASD_training"
VAL_IMG_DIR = DATASET / "LCC_FASD_development"
TEST_IMG_DIR = DATASET / "LCC_FASD_evaluation"

In [17]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train = tf.keras.utils.image_dataset_from_directory(
    TRAIN_IMG_DIR,
    labels="inferred",
    label_mode="binary",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)

valid = tf.keras.utils.image_dataset_from_directory(
    VAL_IMG_DIR,
    labels="inferred",
    label_mode="binary",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test = tf.keras.utils.image_dataset_from_directory(
    TEST_IMG_DIR,
    labels="inferred",
    label_mode="binary",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

Found 8746 files belonging to 2 classes.
Found 3006 files belonging to 2 classes.
Found 7635 files belonging to 2 classes.


In [28]:
print(train.class_names)
print(valid.class_names)
print(test.class_names)

['real', 'spoof']
['real', 'spoof']
['real', 'spoof']


In [29]:
def count_images(folder):
    folder = Path(folder)
    print(folder.name)

    for cls in ["real", "spoof"]:
        files = list((folder / cls).glob("*"))
        print(cls, len(files))

count_images(TRAIN_IMG_DIR)
count_images(VAL_IMG_DIR)
count_images(TEST_IMG_DIR)

LCC_FASD_training
real 1302
spoof 7444
LCC_FASD_development
real 416
spoof 2590
LCC_FASD_evaluation
real 323
spoof 7312


In [18]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])

In [20]:
antispoof_model_v1 = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(224, 224, 3)),
    tf.keras.layers.Rescaling(1./255),

    data_augmentation,

    tf.keras.layers.Conv2D(32, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(64, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(128, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.GlobalAveragePooling2D(),

    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(1, activation="sigmoid")
])

antispoof_model_v1.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [51]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True
)

In [ ]:
train_labels = []

for _, labels in train:
    train_labels.extend(labels.numpy().astype(int).flatten())

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_labels),
    y=train_labels
)

class_weights = dict(enumerate(class_weights))

print(class_weights)
print(train.class_names)

{0: np.float64(3.358678955453149), 1: np.float64(0.587452982267598)}


In [39]:
history = antispoof_model_v1.fit(
    train,
    validation_data=valid,
    epochs=10,
    class_weight=class_weights
)

Epoch 1/10
274/274 ━━━━━━━━━━━━━━━━━━━━ 100s 361ms/step - accuracy: 0.7303 - loss: 0.5566 - val_accuracy: 0.6364 - val_loss: 0.7485
Epoch 2/10
274/274 ━━━━━━━━━━━━━━━━━━━━ 110s 398ms/step - accuracy: 0.7648 - loss: 0.5054 - val_accuracy: 0.5442 - val_loss: 0.9695
Epoch 3/10
274/274 ━━━━━━━━━━━━━━━━━━━━ 103s 375ms/step - accuracy: 0.7700 - loss: 0.4905 - val_accuracy: 0.7578 - val_loss: 0.5459
Epoch 4/10
274/274 ━━━━━━━━━━━━━━━━━━━━ 108s 391ms/step - accuracy: 0.7935 - loss: 0.4586 - val_accuracy: 0.6417 - val_loss: 0.8515
Epoch 5/10
274/274 ━━━━━━━━━━━━━━━━━━━━ 112s 408ms/step - accuracy: 0.8083 - loss: 0.4300 - val_accuracy: 0.6101 - val_loss: 1.0277
Epoch 6/10
274/274 ━━━━━━━━━━━━━━━━━━━━ 119s 432ms/step - accuracy: 0.8402 - loss: 0.3911 - val_accuracy: 0.7119 - val_loss: 0.7157
Epoch 7/10
274/274 ━━━━━━━━━━━━━━━━━━━━ 118s 429ms/step - accuracy: 0.8470 - loss: 0.3741 - val_accuracy: 0.6766 - val_loss: 0.7793
Epoch 8/10
274/274 ━━━━━━━━━━━━━━━━━━━━ 272s 993ms/step - accuracy: 0.8653 -

In [40]:
loss, accuracy = antispoof_model_v1.evaluate(test)
print(f"acc: {accuracy}, loss: {loss}")

239/239 ━━━━━━━━━━━━━━━━━━━━ 24s 97ms/step - accuracy: 0.7890 - loss: 0.5146
acc: 0.788998007774353, loss: 0.5145618915557861


In [42]:
antispoof_model_v1.save('./antispoof_raw.keras')

In [52]:
transfer = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

transfer.trainable = True

for layer in transfer.layers[:-5]:
    layer.trainable = False

antispoof_model_v2 = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(224, 224, 3)),
    tf.keras.layers.Rescaling(1./255),

    data_augmentation,

    transfer,

    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(64, activation="relu"),
    
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

antispoof_model_v2.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [53]:
antispoof_model_v2.compile( 
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), 
    loss="binary_crossentropy", 
    metrics=["accuracy"] 
)

antispoof_model_v2.summary() 

history = antispoof_model_v2.fit(
    train,
    validation_data=valid,
    epochs=10,
    class_weight=class_weights,
    callbacks=[early_stop]
)

Model: "sequential_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_8 (Rescaling)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_3 (Sequential)       │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_8      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 64)             │        81,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,340,033 (8.93 MB)

 Trainable params: 802,049 (3.06 MB)

 Non-trainable params: 1,537,984 (5.87 MB)

Epoch 1/10
274/274 ━━━━━━━━━━━━━━━━━━━━ 64s 226ms/step - accuracy: 0.7645 - loss: 0.4879 - val_accuracy: 0.8363 - val_loss: 0.7085
Epoch 2/10
274/274 ━━━━━━━━━━━━━━━━━━━━ 63s 229ms/step - accuracy: 0.8340 - loss: 0.3792 - val_accuracy: 0.8164 - val_loss: 0.9235
Epoch 3/10
274/274 ━━━━━━━━━━━━━━━━━━━━ 65s 236ms/step - accuracy: 0.8404 - loss: 0.3401 - val_accuracy: 0.8217 - val_loss: 0.7663
Epoch 4/10
274/274 ━━━━━━━━━━━━━━━━━━━━ 65s 235ms/step - accuracy: 0.8605 - loss: 0.3110 - val_accuracy: 0.8263 - val_loss: 0.8164
Epoch 5/10
274/274 ━━━━━━━━━━━━━━━━━━━━ 65s 237ms/step - accuracy: 0.8834 - loss: 0.2765 - val_accuracy: 0.2345 - val_loss: 7.0180


In [54]:
loss, accuracy = antispoof_model_v2.evaluate(test)
print("accuracy: ", accuracy)
print("loss: ", loss)

239/239 ━━━━━━━━━━━━━━━━━━━━ 33s 136ms/step - accuracy: 0.8616 - loss: 0.6206
accuracy:  0.8615586161613464
loss:  0.6205826997756958


In [57]:
antispoof_model_v2.save('./antispoof_transfer.keras')

In [56]:
from sklearn.metrics import classification_report, confusion_matrix

y_true = []
y_pred = []

for images, labels in test:
    pred = antispoof_model_v1.predict(images)
    pred = (pred.flatten() >= 0.6).astype(int)

    y_true.extend(labels.numpy().astype(int).flatten())
    y_pred.extend(pred)

print(confusion_matrix(y_true, y_pred))

print(classification_report(
    y_true,
    y_pred,
    target_names=test.class_names
))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 198ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step
1/1 ━━━━━

In [55]:
from sklearn.metrics import classification_report, confusion_matrix

y_true = []
y_pred = []

for images, labels in test:
    pred = antispoof_model_v2.predict(images)
    pred = (pred.flatten() >= 0.6).astype(int)

    y_true.extend(labels.numpy().astype(int).flatten())
    y_pred.extend(pred)

print(confusion_matrix(y_true, y_pred))

print(classification_report(
    y_true,
    y_pred,
    target_names=test.class_names
))

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 569ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 145ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 147ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 132ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 137ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 141ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 131ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 

In [58]:
base = tf.keras.models.load_model("./antispoof_raw.keras")
transfer = tf.keras.models.load_model("./antispoof_transfer.keras")

In [59]:
loss, accuracy = base.evaluate(test)
print("base test accuracy:", accuracy)
print("base test loss:", loss)

loss, accuracy = transfer.evaluate(test)
print("transfer test accuracy:", accuracy)
print("transfer test loss:", loss)

239/239 ━━━━━━━━━━━━━━━━━━━━ 24s 99ms/step - accuracy: 0.7890 - loss: 0.5146
base test accuracy: 0.788998007774353
base test loss: 0.5145618915557861
239/239 ━━━━━━━━━━━━━━━━━━━━ 34s 138ms/step - accuracy: 0.8616 - loss: 0.6206
transfer test accuracy: 0.8615586161613464
transfer test loss: 0.6205826997756958
